# Opus-as-Grader × V10 Rubric — C-series Analysis

Compares 4 conditions × 2 seeds with grader_backend=opus_subagent against the v10×Qwen-8B C* baseline
(projects/grant_proposal_v2/runs/2026_04_23_v10_qwen8b_foundopt_C{2,3,4}).

Design doc: ~/.claude/plans/polymorphic-sprouting-eagle.md

Key questions:
1. Does Opus grader close the SDPO Goodhart observed under Qwen-8B self-grader (v10 C3: Opus 15→14→11 monotonic regression)?
2. Does C3 (SDPO) still improve rubric_score under Opus? (user core hypothesis)
3. Does the CR-pipeline late-stage drift (v10 C2 Opus 18→15 at iter 10→22) disappear with stronger grader?
4. New attack surface? (6-dim stylistic drift detector)

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path('/home/silas/co-scientist-project')
RUNS_DIR = REPO_ROOT / 'projects/grant_proposal_v2/runs'

# Candidate Opus runs (populated as they complete)
OPUS_CONDITIONS = ['C2', 'C3', 'C4', 'HER']
OPUS_SEEDS = [0, 1]

# Qwen-8B baseline (existing)
QWEN_BASELINE = {
    'C2': RUNS_DIR / '2026_04_23_v10_qwen8b_foundopt_C2',
    'C3': RUNS_DIR / '2026_04_23_v10_qwen8b_foundopt_C3',
    'C4': RUNS_DIR / '2026_04_23_v10_qwen8b_foundopt_C4',
}


def opus_run_path(cond: str, seed: int) -> Path | None:
    """Return the most recent run dir for this condition/seed, or None if not found."""
    pat = f'*_opus_v10_qwen8b_{cond.lower()}_seed{seed}'
    matches = sorted(RUNS_DIR.glob(pat))
    return matches[-1] if matches else None


for cond in OPUS_CONDITIONS:
    for seed in OPUS_SEEDS:
        p = opus_run_path(cond, seed)
        print(f'{cond} seed={seed}: {p}')

for cond, p in QWEN_BASELINE.items():
    print(f'{cond} Qwen-8B baseline: {p} exists={p.exists()}')

## 1. Load buffer + metrics + depth_audit per run

In [ ]:
def load_buffer(run_dir: Path) -> pd.DataFrame:
    p = run_dir / 'buffer.jsonl'
    if not p.exists():
        return pd.DataFrame()
    rows = [json.loads(l) for l in p.read_text().splitlines()]
    return pd.DataFrame(rows)


def load_metrics(run_dir: Path) -> pd.DataFrame:
    p = run_dir / 'metrics.jsonl'
    if not p.exists():
        return pd.DataFrame()
    rows = [json.loads(l) for l in p.read_text().splitlines()]
    return pd.DataFrame(rows)


def load_depth_audit(run_dir: Path) -> pd.DataFrame:
    p = run_dir / 'depth_audit_mid.jsonl'
    if not p.exists():
        return pd.DataFrame()
    rows = [json.loads(l) for l in p.read_text().splitlines()]
    return pd.DataFrame(rows)


all_runs = {}
for cond in OPUS_CONDITIONS:
    for seed in OPUS_SEEDS:
        p = opus_run_path(cond, seed)
        if p is not None and p.exists():
            all_runs[f'{cond}_seed{seed}'] = {
                'path': p,
                'buffer': load_buffer(p),
                'metrics': load_metrics(p),
                'depth_audit': load_depth_audit(p),
            }

for cond, p in QWEN_BASELINE.items():
    all_runs[f'{cond}_qwen_baseline'] = {
        'path': p,
        'buffer': load_buffer(p),
        'metrics': load_metrics(p),
        'depth_audit': load_depth_audit(p),
    }

for k, v in all_runs.items():
    print(f'{k}: buffer rows={len(v["buffer"])}, metrics={len(v["metrics"])}, depth_audit={len(v["depth_audit"])}')

## 2. Trajectory: buffer_max (rubric score) vs iter

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
colors = {'opus_seed0': 'tab:blue', 'opus_seed1': 'tab:cyan', 'qwen': 'tab:orange'}

for ax, cond in zip(axes, OPUS_CONDITIONS):
    for seed in OPUS_SEEDS:
        key = f'{cond}_seed{seed}'
        if key not in all_runs:
            continue
        metrics = all_runs[key]['metrics']
        if not metrics.empty and 'buffer_max' in metrics.columns:
            ax.plot(metrics['iteration'], metrics['buffer_max'],
                    label=f'Opus seed={seed}', color=colors[f'opus_seed{seed}'])
    qkey = f'{cond}_qwen_baseline'
    if qkey in all_runs:
        qm = all_runs[qkey]['metrics']
        if not qm.empty and 'buffer_max' in qm.columns:
            ax.plot(qm['iteration'], qm['buffer_max'],
                    label='Qwen-8B baseline', color=colors['qwen'], linestyle='--')
    ax.set_title(f'Condition {cond}')
    ax.set_xlabel('iter')
    ax.legend(loc='lower right', fontsize=8)
axes[0].set_ylabel('buffer_max (rubric score)')
plt.tight_layout()
plt.show()

## 3. Opus-depth-audit trajectory (ground-truth oracle)

Opus-depth-audit runs independent 4-dim Opus judgement at iter 5/10/15/20/25 via existing
`opus_eval_agent.py` or a parallel Claude subagent (spec in
`src/co_scientist/grant_proposal/opus_eval_agent.py`).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, cond in zip(axes, OPUS_CONDITIONS):
    for seed in OPUS_SEEDS:
        key = f'{cond}_seed{seed}'
        if key not in all_runs:
            continue
        da = all_runs[key]['depth_audit']
        if da.empty:
            continue
        ax.plot(da['iter'], da['total'], 'o-',
                label=f'Opus seed={seed}', color=colors[f'opus_seed{seed}'])
    qkey = f'{cond}_qwen_baseline'
    if qkey in all_runs and not all_runs[qkey]['depth_audit'].empty:
        qd = all_runs[qkey]['depth_audit']
        ax.plot(qd['iter'], qd['total'], 's--', color=colors['qwen'], label='Qwen baseline')
    ax.set_title(f'{cond}  (Opus depth_audit /40)')
    ax.set_xlabel('iter'); ax.set_ylim(0, 40)
    ax.axhline(16, color='gray', alpha=0.3, linestyle=':')
    ax.legend(fontsize=8)
axes[0].set_ylabel('Opus depth_audit total /40')
plt.tight_layout(); plt.show()

## 4. Goodhart detector: rubric_score vs depth_audit divergence

In [ ]:
rows = []
for key, run in all_runs.items():
    m = run['metrics']
    da = run['depth_audit']
    if m.empty or da.empty:
        continue
    # Align on iter
    for _, aud in da.iterrows():
        it = aud['iter']
        mm = m[m['iteration'] == it]
        if mm.empty:
            continue
        rows.append({
            'run': key, 'iter': it,
            'buffer_max': float(mm['buffer_max'].iloc[0]) if 'buffer_max' in mm.columns else np.nan,
            'opus_total_40': aud.get('total', np.nan),
        })
goodhart_df = pd.DataFrame(rows)
if not goodhart_df.empty:
    fig, ax = plt.subplots(figsize=(6, 6))
    for run, grp in goodhart_df.groupby('run'):
        ax.plot(grp['buffer_max'], grp['opus_total_40'], 'o-', label=run, alpha=0.7)
    ax.set_xlabel('Qwen/Opus rubric_score (buffer_max)')
    ax.set_ylabel('Opus depth_audit /40')
    ax.axline((0.5, 20), slope=40, color='gray', alpha=0.3, linestyle=':')
    ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1))
    plt.tight_layout(); plt.show()
    display(goodhart_df)
else:
    print('No aligned rubric/depth-audit data yet.')

## 5. 6-dim stylistic drift detector (Opus attack-surface check)

Flag if iter-final plan drifts >2σ from iter-0 baseline on: char count, citation density,
math symbol density, hedging phrase freq, LaTeX formality, avg sentence length.

In [ ]:
import re

HEDGE_PHRASES = [r'\bwe expect\b', r'\bwe aim to\b', r'\bshould\b', r'\bmay\b', r'\bcould\b',
                 r'\blikely\b', r'\banticipate\b', r'\bhope to\b']
LATEX_RE = re.compile(r'\\[A-Za-z]+|\$[^$]*\$|\\begin\{')
MATH_SYM_RE = re.compile(r'[∇∑∏∫∂αβγδεθλμσφωΔΣΠΩ]|\\mathcal|\\mathbb|\\nabla')
CITATION_RE = re.compile(r'\([A-Z][A-Za-z-]+(?:\s+et\s+al\.)?,\s*\d{4}')


def style_metrics(text: str) -> dict:
    if not text:
        return {k: 0.0 for k in ['chars', 'cite_density', 'math_sym_density', 'hedge_freq', 'latex_density', 'avg_sent_len']}
    sents = re.split(r'[.!?]+', text)
    sents = [s for s in sents if s.strip()]
    n_words = max(1, len(text.split()))
    hedge_count = sum(len(re.findall(h, text, flags=re.IGNORECASE)) for h in HEDGE_PHRASES)
    return {
        'chars': len(text),
        'cite_density': len(CITATION_RE.findall(text)) / n_words * 1000,
        'math_sym_density': len(MATH_SYM_RE.findall(text)) / n_words * 1000,
        'hedge_freq': hedge_count / n_words * 1000,
        'latex_density': len(LATEX_RE.findall(text)) / n_words * 1000,
        'avg_sent_len': (sum(len(s.split()) for s in sents) / len(sents)) if sents else 0.0,
    }


drift_rows = []
for key, run in all_runs.items():
    buf = run['buffer']
    if buf.empty or 'plan_text' not in buf.columns:
        continue
    iter_groups = buf.groupby('iteration' if 'iteration' in buf.columns else 'iter')
    for it, grp in iter_groups:
        best_row = grp.loc[grp['aggregate_reward'].idxmax()] if 'aggregate_reward' in grp.columns else grp.iloc[0]
        m = style_metrics(best_row.get('plan_text', ''))
        m['run'] = key; m['iter'] = int(it)
        drift_rows.append(m)
drift_df = pd.DataFrame(drift_rows)
if not drift_df.empty:
    display(drift_df.groupby('run').agg({c: ['first', 'last'] for c in drift_df.columns if c not in ['run', 'iter']}))
else:
    print('No plan data yet.')

## 6. Pre-registered decision check (from plan §4)

- **C3-opus iter 22 Opus-depth-audit ≥ 14/40**: validates Opus grader removes SDPO Goodhart.
- **C3-opus rubric_score Δ vs C2-opus at iter 22 ≥ +0.03**: validates SDPO still improves rubric under Opus.
- **Any RL > C2-opus by ≥ 2 points at iter 22**: RL provides substance gain.
- **Refuted if**: any RL drops ≥ 3 points iter 10 → iter 22, OR C3-opus < 14/40.

In [ ]:
def depth_at(run_key: str, target_iter: int, tol: int = 2) -> float | None:
    if run_key not in all_runs:
        return None
    da = all_runs[run_key]['depth_audit']
    if da.empty:
        return None
    near = da[(da['iter'] >= target_iter - tol) & (da['iter'] <= target_iter + tol)]
    if near.empty:
        return None
    return float(near.loc[near['iter'].idxmax(), 'total'])


summary_rows = []
for cond in OPUS_CONDITIONS:
    for seed in OPUS_SEEDS:
        key = f'{cond}_seed{seed}'
        summary_rows.append({
            'run': key,
            'opus_iter_10': depth_at(key, 10),
            'opus_iter_22': depth_at(key, 22),
            'delta_iter_10_22': (depth_at(key, 22) or 0) - (depth_at(key, 10) or 0) if depth_at(key, 10) else None,
        })
summary = pd.DataFrame(summary_rows)
display(summary)

# Pre-registered checks
print()
print('=== Pre-registered decision ===')
for _, row in summary.iterrows():
    run = row['run']
    ovr22 = row['opus_iter_22']
    delta = row['delta_iter_10_22']
    if pd.isna(ovr22) or ovr22 is None:
        print(f'{run}: no iter 22 data yet')
        continue
    flags = []
    if run.startswith('C3') and ovr22 < 14:
        flags.append('REFUTED: C3 < 14/40')
    if delta is not None and delta <= -3:
        flags.append(f'REFUTED: regression {delta:+.1f}')
    print(f'{run}: Opus/40={ovr22:.1f}  Δ10→22={delta}  flags={flags or "OK"}')